In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
EMB_CSV = "/content/drive/MyDrive/models_india_2015/india2015_cluster_embeddings_random20k_resnet18_proxy_nl.csv"
CLUSTERS_CSV = "/content/drive/MyDrive/IN2015_clusters_nightlights.csv"

OUT_DIR = "/content/drive/MyDrive/models_india_2015"
PRED_OUT_CSV = f"{OUT_DIR}/india2015_cluster_predictions_random20k_proxy.csv"

TARGET_COL = "target"  # DHS wealth index (cluster-level)

RANDOM_SEED = 42
N_SPLITS_RANDOM_CV = 5
N_SPLITS_SPATIAL_CV = 5
N_SPATIAL_GROUPS = 25    # KMeans geographic blocks
RIDGE_ALPHA = 10.0

print("EMB_CSV:", EMB_CSV)
print("CLUSTERS_CSV:", CLUSTERS_CSV)
print("OUT:", PRED_OUT_CSV)

In [ ]:
import numpy as np
import pandas as pd

clusters = pd.read_csv(CLUSTERS_CSV)
emb_df = pd.read_csv(EMB_CSV)

clusters["cluster_id"] = clusters["cluster_id"].astype(int)
emb_df["cluster_id"] = emb_df["cluster_id"].astype(int)

keep_cols = ["cluster_id","cluster_lat", "cluster_lon", TARGET_COL, "nightlights_mean"]

df = emb_df.merge(clusters[keep_cols], on="cluster_id", how="left")
df.head()

In [ ]:
before = len(df)
df = df[df["cluster_lon"].between(60, 100) & df["cluster_lat"].between(5, 40)].copy()
df = df.reset_index(drop=True)
print(f"Dropped {before-len(df)} rows with invalid coords; remaining {len(df)}")

In [ ]:
y = df[TARGET_COL].to_numpy(dtype=float)
mask = np.isfinite(y)

df_common = df.loc[mask].reset_index(drop=True)

In [ ]:
from sklearn.model_selection import KFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.cluster import KMeans

def cv_regression_random(df_in, feat_cols, y_col, alpha=10.0, n_splits=5, seed=42):
    X = df_in[feat_cols].to_numpy(dtype=np.float32)
    y = df_in[y_col].to_numpy(dtype=float)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    preds = np.full(len(y), np.nan)
    r2s, maes = [], []

    for tr, va in kf.split(X):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ])
        pipe.fit(X[tr], y[tr])
        yhat = pipe.predict(X[va])

        preds[va] = yhat
        r2s.append(r2_score(y[va], yhat))
        maes.append(mean_absolute_error(y[va], yhat))

    return {
        "alpha": float(alpha),
        "r2_mean": float(np.mean(r2s)),
        "r2_std": float(np.std(r2s)),
        "mae_mean": float(np.mean(maes)),
        "mae_std": float(np.std(maes)),
        "oof_pred": preds,
    }

def make_spatial_groups(df_in, n_groups=25, seed=42):
    coords = df_in[["cluster_lat","cluster_lon"]].to_numpy(dtype=float)
    km = KMeans(n_clusters=n_groups, random_state=seed, n_init="auto")
    return km.fit_predict(coords)

def cv_regression_spatial(df_in, feat_cols, y_col, groups, alpha=10.0, n_splits=5):
    X = df_in[feat_cols].to_numpy(dtype=np.float32)
    y = df_in[y_col].to_numpy(dtype=float)

    gkf = GroupKFold(n_splits=n_splits)
    preds = np.full(len(y), np.nan)
    r2s, maes = [], []

    for tr, va in gkf.split(X, y, groups=groups):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ])
        pipe.fit(X[tr], y[tr])
        yhat = pipe.predict(X[va])

        preds[va] = yhat
        r2s.append(r2_score(y[va], yhat))
        maes.append(mean_absolute_error(y[va], yhat))

    return {
        "alpha": float(alpha),
        "r2_mean": float(np.mean(r2s)),
        "r2_std": float(np.std(r2s)),
        "mae_mean": float(np.mean(maes)),
        "mae_std": float(np.std(maes)),
        "oof_pred": preds,
    }

In [ ]:
# Random CV
alpha = RIDGE_ALPHA

res_emb_r = cv_regression_random(
    df_common, feat_cols, TARGET_COL,
    alpha=alpha,
    n_splits=N_SPLITS_RANDOM_CV,
    seed=RANDOM_SEED
)
df_common["pred_randomcv_emb"] = res_emb_r["oof_pred"]

print("R2 mean/std:", res_emb_r["r2_mean"], res_emb_r["r2_std"])
print("MAE mean/std:", res_emb_r["mae_mean"], res_emb_r["mae_std"])

In [ ]:
# Spatial CV

groups = make_spatial_groups(df_common, n_groups=N_SPATIAL_GROUPS, seed=RANDOM_SEED)
df_common["spatial_group"] = groups.astype(int)

res_emb_s = cv_regression_spatial(
    df_common, feat_cols, TARGET_COL,
    groups=df_common["spatial_group"].to_numpy(),
    alpha=RIDGE_ALPHA,
    n_splits=N_SPLITS_SPATIAL_CV
)
df_common["pred_spatialcv_emb"] = res_emb_s["oof_pred"]

print("R2 mean/std:", res_emb_s["r2_mean"], res_emb_s["r2_std"])
print("MAE mean/std:", res_emb_s["mae_mean"], res_emb_s["mae_std"])

In [ ]:
summary = pd.DataFrame([
    {"model": "Embedding", "cv": "Random",  "r2_mean": res_emb_r["r2_mean"], "mae_mean": res_emb_r["mae_mean"]},
    {"model": "Embedding", "cv": "Spatial", "r2_mean": res_emb_s["r2_mean"], "mae_mean": res_emb_s["mae_mean"]},
])

summary

In [ ]:
out_cols = ["cluster_id", "cluster_lat", "cluster_lon", TARGET_COL, "nightlights_mean", "spatial_group", "pred_randomcv_emb", "pred_spatialcv_emb"]

pred_out = df_common[out_cols].copy()
pred_out.to_csv(PRED_OUT_CSV, index=False)

pred_out.head()

## Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error

def scatter_true_pred(ax, y_true, y_pred, title):
    m = np.isfinite(y_pred) & np.isfinite(y_true)
    yt = y_true[m]
    yp = y_pred[m]

    r2 = r2_score(yt, yp)
    mae = mean_absolute_error(yt, yp)

    ax.scatter(yt, yp, s=10, alpha=0.5)
    lo = min(yt.min(), yp.min())
    hi = max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi])

    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.set_title(f"{title}\nR2={r2:.3f}, MAE={mae:.3f}")
    ax.grid(True)

# Random vs Spatial
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))

scatter_true_pred(
    axes[0],
    df_common[TARGET_COL].to_numpy(),
    df_common["pred_randomcv_emb"].to_numpy(),
    "Random CV (Embedding)"
)

scatter_true_pred(
    axes[1],
    df_common[TARGET_COL].to_numpy(),
    df_common["pred_spatialcv_emb"].to_numpy(),
    "Spatial CV (Embedding)"
)

plt.suptitle("India DHS 2015 — Ridge on trained Embeddings", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
pred_plot = pred.copy()

pred_col = "pred_spatialcv_emb"

pred_plot["poverty_pred"] = -pred_plot[pred_col].astype(float)

lo = np.nanmin(pred_plot["poverty_pred"].to_numpy(dtype=float))
hi = np.nanmax(pred_plot["poverty_pred"].to_numpy(dtype=float))
pred_plot["poverty01"] = (pred_plot["poverty_pred"] - lo) / (hi - lo + 1e-9)

pred_plot = pred_plot[
    pred_plot["cluster_lon"].between(60, 100) &
    pred_plot["cluster_lat"].between(5, 40)
].copy()

plt.figure(figsize=(8, 8))
sc = plt.scatter(
    pred_plot["cluster_lon"], pred_plot["cluster_lat"],
    c=pred_plot["poverty01"],
    s=12, alpha=0.85
)
plt.colorbar(sc, label="Predicted poverty (0 = less poor, 1 = poorer)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("India DHS 2015 — Predicted Poverty Map (Proxy-CNN Embedding, Spatial CV)")

plt.xlim(66, 98)
plt.ylim(6, 37)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
!pip -q install cartopy

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pred = pd.read_csv(PRED_OUT_CSV)

pred = pred[
    pred["cluster_lon"].between(68, 98) &
    pred["cluster_lat"].between(6, 37)
].copy().reset_index(drop=True)

pred["poverty_true"] = -pred["target"].astype(float)
pred["poverty_emb_rand"] = -pred["pred_randomcv_emb"].astype(float)
pred["poverty_emb_spat"] = -pred["pred_spatialcv_emb"].astype(float)

def norm01(x, lo, hi):
    x = np.asarray(x, dtype=float)
    return (x - lo) / (hi - lo + 1e-9)

vals = [
    pred["poverty_true"],
    pred["poverty_emb_rand"],
    pred["poverty_emb_spat"],
]

lo = np.nanmin(np.concatenate([v.to_numpy() for v in vals]))
hi = np.nanmax(np.concatenate([v.to_numpy() for v in vals]))

# normalize to 0–1
pred["true01"]     = norm01(pred["poverty_true"],     lo, hi)
pred["emb_rand01"] = norm01(pred["poverty_emb_rand"], lo, hi)
pred["emb_spat01"] = norm01(pred["poverty_emb_spat"], lo, hi)

cols = ["true01", "emb_rand01", "emb_spat01"]
titles = ["True poverty (DHS wealth)", "Predicted (Embedding, Random CV)", "Predicted (Embedding, Spatial CV)"]

extent = [68, 98, 6, 37]
fig, axs = plt.subplots(1, len(cols), figsize=(5.2*len(cols), 6),
                        subplot_kw={'projection': ccrs.PlateCarree()})

for ax, col, title in zip(axs, cols, titles):
    ax.set_extent(extent)
    ax.add_feature(cfeature.LAND, alpha=0.15)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.8)
    try:
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3, alpha=0.6)
    except:
        pass

    sc = ax.scatter(
        pred["cluster_lon"], pred["cluster_lat"],
        c=pred[col], s=10, alpha=0.85,
        transform=ccrs.PlateCarree()
    )
    ax.set_title(title)

    cb = plt.colorbar(sc, ax=ax, shrink=0.75, pad=0.02)
    cb.set_label("Poverty (0=less poor, 1=poorer)")

plt.suptitle("India DHS 2015 — Poverty Mapping", fontsize=16)
plt.tight_layout()
plt.show()